In [1]:
import os
from pathlib import Path

import mlflow
from sklearn.feature_extraction.text import TfidfVectorizer

from support_assistant.data import load_intents
from support_assistant.model import build_pipeline, score_predictions, split_intents

ROOT = Path.cwd().parents[2]
os.chdir(ROOT)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("banking77-intents")

df = load_intents("data/raw/banking77_train.csv")
print(Path.cwd().name, "|", len(df), "rows")

2026/10/07 15:34:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/07 15:34:45 INFO mlflow.store.db.utils: Updating database tables
2026/10/07 15:34:45 INFO mlflow.tracking.fluent: Experiment with name 'banking77-intents' does not exist. Creating a new experiment.


support-assistant | 10003 rows


In [2]:
train_df, val_df = split_intents(df)
pipeline = build_pipeline("logistic")

with mlflow.start_run(run_name="by-hand-single-split"):
    mlflow.log_param("classifier", "logistic")
    mlflow.log_param("evaluation","one split, seed 42")

    pipeline.fit(train_df["text"], train_df["category"])
    scores = score_predictions(val_df["category"], pipeline.predict(val_df["text"]))

    mlflow.log_metric("macro_f1", scores["macro_f1"])
    mlflow.log_metric("accuracy", scores["accuracy"])
    mlflow.sklearn.log_model(pipeline, name="model")

print(scores)

2026/10/07 15:41:00 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/mac/Desktop/2026/Bootcamp/code/support-assistant
2026/10/07 15:41:08 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/mac/Desktop/2026/Bootcamp/code/support-assistant
2026/10/07 15:41:08 INFO mlflow.utils.environment: Detected uv project at /Users/mac/Desktop/2026/Bootcamp/code/support-assistant. Attempting to export requirements via 'uv export'.
2026/10/07 15:41:08 INFO mlflow.utils.uv_utils: Exported 25 dependencies via uv
2026/10/07 15:41:08 INFO mlflow.utils.environment: Successfully exported 25 requirements from uv project. Skipping package capture based inference.
2026/10/07 15:41:08 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{'accuracy': 0.8605, 'macro_f1': 0.858317799571448}


In [3]:
pairs = TfidfVectorizer(ngram_range=(1, 2)).fit(["my top up failed", "top up not working"])
print(list(pairs.get_feature_names_out()))

tuned = build_pipeline("logistic").set_params(tfidf__ngram_range=(1, 2), classifier__C=10.0)
print(tuned.named_steps["tfidf"].ngram_range, tuned.named_steps["classifier"].C)

['failed', 'my', 'my top', 'not', 'not working', 'top', 'top up', 'up', 'up failed', 'up not', 'working']
(1, 2) 10.0


In [4]:
runs = mlflow.search_runs(experiment_names=["banking77-intents"])
print(len(runs), "runs |", len(runs.columns), "columns")

columns = {
    "tags.mlflow.runName": "run",
    "metrics.macro_f1": "macro_f1",
    "metrics.macro_f1_std": "std",
    "metrics.accuracy": "accuracy",
    "metrics.fit_seconds": "fit_seconds",
}
table = runs.rename(columns=columns)[list(columns.values())].sort_values("macro_f1", ascending=False)
table.round(4)

13 runs | 26 columns


,run,macro_f1,std,accuracy,fit_seconds
9,logistic-ngram1-C10.0,0.8817,0.0046,0.8832,0.8965
4,linear_svc-ngram1-C1.0,0.8795,0.0044,0.8822,0.3272
1,linear_svc-ngram2-C1.0,0.8779,0.0030,0.8800,0.3784
6,logistic-ngram2-C10.0,0.8775,0.0040,0.8794,5.5908
0,linear_svc-ngram2-C10.0,0.8754,0.0046,0.8773,0.6932
3,linear_svc-ngram1-C10.0,0.8612,0.0040,0.8645,0.4233
12,by-hand-single-split,0.8583,NaN,0.8605,NaN
10,logistic-ngram1-C1.0,0.8539,0.0104,0.8591,0.5756
5,linear_svc-ngram1-C0.1,0.8456,0.0105,0.8544,0.3164
2,linear_svc-ngram2-C0.1,0.8407,0.0073,0.8489,0.3269


In [5]:
grid_runs = runs.dropna(subset=["params.classifier__C"])
grid_runs.pivot_table(
    index=["params.classifier", "params.tfidf__ngram_range"],
    columns="params.classifier__C",
    values="metrics.macro_f1",
).round(4)

params.classifier__C                            0.1     1.0    10.0
params.classifier params.tfidf__ngram_range                        
linear_svc        (1, 1)                     0.8456  0.8795  0.8612
                  (1, 2)                     0.8407  0.8779  0.8754
logistic          (1, 1)                     0.6643  0.8539  0.8817
                  (1, 2)                     0.6056  0.8361  0.8775

In [6]:
best = runs.sort_values("metrics.macro_f1", ascending=False).iloc[0]
print(best["tags.mlflow.runName"], "|", round(best["metrics.macro_f1"], 4), "|", best["tags.model_uri"])

model = mlflow.sklearn.load_model(best["tags.model_uri"])
messages = ["I think someone has taken my card", "the cash machine ate my card", "top up didn't go through"]
for message, intent in zip(messages, model.predict(messages)):
    print(f"{intent:26} {message}")

logistic-ngram1-C10.0 | 0.8817 | models:/m-79554ee7114441b38170b5a1f1d417d8
compromised_card           I think someone has taken my card
card_swallowed             the cash machine ate my card
top_up_failed              top up didn't go through
